# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuguda999/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Both picked from the ML appendix — the paper itself flags that section as "exploratory... do not override direct portfolio evidence" (Study Scope & Evidence Standard, p.4), so these are the right place to ask harder methodology questions; the headline aggregate findings (growth anatomy, performance curve, click capture) are direct comparisons on large disclosed samples and don't raise the same flags.

**Finding A — "What Predicts Growth?" (p.29, Logistic Regression, 71% holdout accuracy).**
The trend label is defined on p.5 as "30d-vs-prev-30d impression change" (up/down/stable/flat/new) — a same-period proxy, structurally similar to my own starter-CSV label from ML-02/ML-03. The methodology page confirms an "80/20 split" but doesn't say whether that split was random by content row or grouped by brand. With 57 brands feeding into one page-level model, a random row split risks the exact thing I measured directly in section 2 below: my own Random Forest's precision@10 fell from 0.900 (naive random split) to 0.300 (grouped by client) on the same features and label shape. **My question:** was the 80/20 split grouped by brand, and if not, would a brand-grouped holdout number look different from 71%? I'm not claiming it would — I'm asking because I have a measured example, on a similar page-level/brand-grouped setup, where the gap was large.

**Finding B — "What Predicts Health?" (p.27, Random Forest feature importance).**
The paper already discloses the risk here, which I want to be clear about: "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal." Health score = impressions(30) + position(30) + CTR(20) + scroll(20) — and Average Position (43%), Impressions (32%), and CTR (8%) are the top three importances, together over 80%. That's my own skill's leakage taxonomy #1 (label-derived features) — already named by the paper, not something I'm "catching." **My question:** would a train-with/train-without comparison (fit once with position/impressions/CTR, once without) show how much of that 83% importance collapses to near-zero versus how much real signal remains in content age, word count, days visible, and AI sessions? That test would turn "importance is descriptive, not causal" into a measured number instead of a caveat.

In [1]:
import pandas as pd

paper_audit = pd.DataFrame([
    {
        "finding": "A — What Predicts Growth? (p.29)",
        "method": "Logistic Regression, 80/20 split, 71% holdout accuracy",
        "label_source": "30d-vs-prev-30d impression change (p.5) — same-period proxy",
        "my_question": "Was the 80/20 split grouped by brand or random by row? "
                        "My own grouped-vs-random test (section 2) showed a large gap on a similar setup.",
    },
    {
        "finding": "B — What Predicts Health? (p.27)",
        "method": "Random Forest feature importance on health score",
        "label_source": "Health score = impressions(30) + position(30) + CTR(20) + scroll(20) "
                         "— paper already discloses partial construction overlap",
        "my_question": "Would a train-with/train-without test on position/impressions/CTR show how much "
                        "of the 83% combined importance is the model re-deriving its own scoring formula?",
    },
])
paper_audit

,finding,method,label_source,my_question
0,A — What Predicts Growth? (p.29),"Logistic Regression, 80/20 split, 71% holdout ...",30d-vs-prev-30d impression change (p.5) — same...,Was the 80/20 split grouped by brand or random...
1,B — What Predicts Health? (p.27),Random Forest feature importance on health score,Health score = impressions(30) + position(30) ...,Would a train-with/train-without test on posit...


## 2. My model under an honest split (before/after)

**Before:** a naive random row split on my ML-08 Random Forest — 42 of 44 clients leak across train/test (same client's rows on both sides), letting the model partly memorize the client instead of learning the pattern.

**After:** the grouped split from ML-08 (`GroupShuffleSplit` on `client_hash_id`, zero client overlap).

Same features, same model, same test-set size ratio — only the split changes. The gap between the two IS the finding: how much of the "before" score was memorization, not signal.

In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# find the repo root from wherever this kernel started (VS Code/Colab/CLI all differ)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET http_timeout=300")
con.execute("SET http_retries=5")
con.execute("SET http_retry_wait_ms=1000")
con.execute("SET http_retry_backoff=2")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# same feature build + proxy label as ML-04/ML-07/ML-08
feat = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_prev,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_prev,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_prev,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN report_date END) AS active_days_prev,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last
    FROM read_parquet('{MONTH}')
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) > 0
""").df()

feat["ctr_prev"] = feat["clk_prev"] / feat["imp_prev"]
feat["is_declining"] = (feat["imp_last"] < 0.8 * feat["imp_prev"]).astype(int)
feat = feat.dropna(subset=["avg_position_prev"]).reset_index(drop=True)
feat["log_imp_prev"] = np.log1p(feat["imp_prev"])
feat["log_clk_prev"] = np.log1p(feat["clk_prev"])

FEATURES = ["log_imp_prev", "log_clk_prev", "ctr_prev", "avg_position_prev", "active_days_prev"]
X = feat[FEATURES]
y = feat["is_declining"]
groups = feat["client_hash_id"]
print(f"shape: {feat.shape[0]:,} rows  |  clients: {feat['client_hash_id'].nunique()}  |  base rate: {y.mean():.3f}")

shape: 150,675 rows  |  clients: 44  |  base rate: 0.326


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


def fit_eval(X_tr, X_te, y_tr, y_te):
    model = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    p = model.predict_proba(X_te)[:, 1]
    row = {"test_auc": round(roc_auc_score(y_te, p), 3)}
    for k in [10, 20, 50, 100]:
        row[f"precision@{k}"] = round(precision_at_k(p, y_te, k), 3)
    return row


# BEFORE: naive random row split — dishonest, same client can land on both sides
X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(X, y, feat.index, test_size=0.25, random_state=42, stratify=y)
overlap_before = set(feat.loc[idx_tr, "client_hash_id"]) & set(feat.loc[idx_te, "client_hash_id"])
before = fit_eval(X_tr, X_te, y_tr, y_te)
before["client_overlap"] = len(overlap_before)

# AFTER: grouped split by client_hash_id — honest, zero overlap
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
after = fit_eval(X.iloc[tr_idx], X.iloc[te_idx], y.iloc[tr_idx], y.iloc[te_idx])
after["client_overlap"] = len(set(feat.iloc[tr_idx]["client_hash_id"]) & set(feat.iloc[te_idx]["client_hash_id"]))

comparison = pd.DataFrame([{"split": "BEFORE (random row split)", **before},
                            {"split": "AFTER (grouped by client)", **after}])
comparison

,split,test_auc,precision@10,precision@20,precision@50,precision@100,client_overlap
0,BEFORE (random row split),0.663,0.9,0.8,0.78,0.73,42
1,AFTER (grouped by client),0.578,0.3,0.3,0.44,0.47,0


**Reading the gap:** the naive split leaked 42 of 44 clients across train/test and precision@10 read 0.900 — the grouped split (0 client overlap) drops it to 0.300. That 0.6 swing at K=10 is memorization, not skill: with most clients visible in training, the model could partly recognize "this is client X's typical page" rather than learn the CTR/volume pattern. The grouped number (0.300 at K=10, matching ML-08) is the one I stand behind; the random-split number was never a real result.

## 3. Leakage audit

The attack checklist, run against my final ML-08 feature set (`log_imp_prev`, `log_clk_prev`, `ctr_prev`, `avg_position_prev`, `active_days_prev`), on the honest grouped split:

- [x] **Timeline drawn:** every feature is aggregated over days 1–15 only; the label (`is_declining`) is computed from days 16–31 only. No overlap.
- [x] **No label-derived/sibling columns in the features** — verified below with the train-with/train-without test.
- [x] **No product flags as features** — none used; every column is a raw GSC observation (impressions, clicks, position), never a FlyRank score/flag.
- [x] **Population selection checked for outcome-window info:** I keep a row only if `imp_prev > 0` — that filter uses ONLY the feature window (days 1–15), never the label window. Disclosed, not hidden.
- [x] **Split grouped by the repeating entity** — `client_hash_id`, zero overlap (section 2).
- [x] **Base rate printed next to every metric** — done throughout ML-07/ML-08/section 2.
- [x] **Top feature importance sanity-checked** — ML-08 permutation importance topped out around 0.035 (AUC drop), nowhere near "suspiciously perfect."
- [x] **Metrics recomputed out-of-fold** — every number above and in ML-08 is test-set only, never in-sample.
- [ ] **Sealed/holdout claims** — N/A, I have never claimed a sealed holdout. Worth saying explicitly: this is a grouped test split, not a sealed one, and `month=2026-03` is a mid-panel month I've iterated on directly — not a blind evaluation.

**The confession test** (per the skill: add the suspect, watch it jump toward 1.0, remove it):

In [4]:
# WITHOUT the suspect (final honest feature set, same grouped split as section 2)
auc_without = fit_eval(X.iloc[tr_idx], X.iloc[te_idx], y.iloc[tr_idx], y.iloc[te_idx])["test_auc"]
print(f"WITHOUT suspect (final feature set): test AUC = {auc_without:.3f}")

# WITH the suspect: the exact future ratio is_declining is computed from
feat["future_ratio_LEAKY"] = feat["imp_last"] / feat["imp_prev"]
FEATURES_LEAKY = FEATURES + ["future_ratio_LEAKY"]
X_leaky = feat[FEATURES_LEAKY]
model_leaky = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
model_leaky.fit(X_leaky.iloc[tr_idx], y.iloc[tr_idx])
auc_with = roc_auc_score(y.iloc[te_idx], model_leaky.predict_proba(X_leaky.iloc[te_idx])[:, 1])
print(f"WITH suspect (+ future_ratio_LEAKY):  test AUC = {auc_with:.3f}  <- the confession")

feat = feat.drop(columns=["future_ratio_LEAKY"])
print(f"\nSuspect removed. AUC collapses from {auc_with:.3f} back to {auc_without:.3f} — "
      f"exactly the pattern the skill describes. My final feature set (without the suspect) "
      f"is clean: no such collapse is possible because the suspect was never in it.")

WITHOUT suspect (final feature set): test AUC = 0.578


WITH suspect (+ future_ratio_LEAKY):  test AUC = 1.000  <- the confession

Suspect removed. AUC collapses from 1.000 back to 0.578 — exactly the pattern the skill describes. My final feature set (without the suspect) is clean: no such collapse is possible because the suspect was never in it.


## 4. Claim rewrite

**My boldest sentence, from `w02_ml_task_framing.ipynb` (ML-03):**

> "On this starter slice, under client-holdout validation, a learned ranking beats the fixed rule: precision@50 = 0.240 (rule) vs 0.680 (random forest) — roughly 2.8x more true positives in the top 50 picks. **That's evidence for using ML here, not just an assumption.**"

That sentence generalizes past its evidence. It was true on the 30,000-row starter CSV, under that dataset's own proxy label — but it doesn't transfer, and I know that now because my own later work contradicts it: in ML-08, on the real March 2026 warehouse slice, under an honest grouped split, **none of three trained models beat my ML-07 baseline rule at any K.** The starter-CSV result was directional evidence for trying ML on THAT slice, not a general claim that "ML beats rules here."

**Rewrite:**

> Observed on the 30,000-row starter CSV only, under client-holdout validation: a random forest measured a higher precision@50 than that dataset's rule baseline (0.680 vs 0.240). This is a directional signal that a model *can* out-rank a hand-written rule on some slices of this data — it is not evidence that it *will*, and it did not hold on the March 2026 warehouse slice I later tested (ML-08: baseline won at every K). The decision-support claim is "worth testing a model against your baseline, on your own data" — not "ML beats rules here."

In [5]:
import json

with open("outputs/model_results.json") as f:
    starter_results = json.load(f)

print("starter CSV (ML-03) baseline precision@50:      ", starter_results["baseline"]["baseline_precision_at_50"])
print("starter CSV (ML-03) random forest precision@50: ", starter_results["models"]["random_forest"]["precision_at_50"])
print()
print("warehouse slice (ML-08 / section 2) comparison, same grouped test split:")
comparison

starter CSV (ML-03) baseline precision@50:       0.24
starter CSV (ML-03) random forest precision@50:  0.68

warehouse slice (ML-08 / section 2) comparison, same grouped test split:


,split,test_auc,precision@10,precision@20,precision@50,precision@100,client_overlap
0,BEFORE (random row split),0.663,0.9,0.8,0.78,0.73,42
1,AFTER (grouped by client),0.578,0.3,0.3,0.44,0.47,0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.